# 06 — YOLO preprocessing, dataset organization and augmentation

This notebook turns the validated archive into the folder layout expected by
Ultralytics YOLOv8 and writes the dataset configuration:

```
data/yolo/
├── images/
│   ├── train/
│   ├── valid/
│   └── test/
├── labels/
│   ├── train/
│   ├── valid/
│   └── test/
├── data.yaml
└── train_config.json
```

The archive only provides `Train` and `Val`, so `Train` becomes the training
set and `Val` is split **50/50** into `valid` and `test` (stratified by class).
This keeps the provided boundary between training data and evaluation data.

The notebook:

1. loads the manifest created by notebook 05 (paired and validated files);
2. recreates the generated `data/yolo/` tree (never touches `data/archive/`);
3. copies each image **with** its annotation into the right split;
4. verifies counts and pairing of the copied files;
5. writes `data.yaml` and the augmentation/training configuration;
6. displays copied samples with their boxes to confirm the transfer.

> **How to read this notebook:** every code cell is preceded by a short
> summary that explains what the step does and how to interpret its output.

## 1. Imports and constants

Imports the copying utilities and defines the input manifest, the output
dataset directory and the validation/test split ratio.

**How to read the output:** the printed paths must point to the manifest
written by notebook 05 and to the new `data/yolo/` dataset.

In [ ]:
from pathlib import Path
import collections
import json
import random
import shutil

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MANIFEST_PATH = PROCESSED_DIR / "yolo_manifest.json"

OUTPUT_DIR = ROOT_DIR / "data" / "yolo"
DATA_YAML = OUTPUT_DIR / "data.yaml"
TRAIN_CONFIG_PATH = OUTPUT_DIR / "train_config.json"

TARGET_SPLITS = ["train", "valid", "test"]
TEST_SIZE = 0.5
SEED = 42

print("Manifest  :", MANIFEST_PATH)
print("Dataset   :", OUTPUT_DIR)
print("Data yaml :", DATA_YAML)

## 2. Load the validated manifest

Reads `yolo_manifest.json` and keeps only the records that passed the format
checks of notebook 05. It also verifies that the class names stored in the
manifest are still the expected ones.

**How to read the output:** `Train` must be 4737 records and `Val` 510. If
`invalid` is greater than `0`, those files are skipped during the copy and
reported at the end of the notebook.

In [ ]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"{MANIFEST_PATH} not found. Run 05_yolo_data_overview.ipynb first."
    )

with open(MANIFEST_PATH) as handle:
    manifest = json.load(handle)

classes = manifest["classes"]
class_to_id = {name: int(i) for name, i in manifest["class_to_id"].items()}
id_to_class = {int(k): v for k, v in manifest["id_to_class"].items()}
assert id_to_class == {i: name for i, name in enumerate(classes)}


def usable_records(split: str):
    return [record for record in manifest["splits"][split] if record["valid"]]


train_records = usable_records("Train")
val_source = usable_records("Val")
invalid_records = [
    record
    for split in manifest["splits"]
    for record in manifest["splits"][split]
    if not record["valid"]
]

print("Classes :", classes)
print("Train   :", len(train_records))
print("Val     :", len(val_source))
print("Invalid :", len(invalid_records))

## 3. Recreate the generated dataset tree

Deletes a previous `data/yolo/` (only that generated folder, never
`data/archive/` or `data/Data/`) and creates the six `images` / `labels`
sub-folders required by YOLO.

**How to read the output:** the list of created directories. Running this cell
again is safe: the generated dataset is rebuilt from the archive every time,
so the copy step is reproducible.

In [ ]:
if OUTPUT_DIR.exists():
    print("Removing previously generated dataset:", OUTPUT_DIR)
    shutil.rmtree(OUTPUT_DIR)

for split in TARGET_SPLITS:
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Created:")
for split in TARGET_SPLITS:
    print(f"  images/{split}  labels/{split}")

## 4. Assign records to train / valid / test

`Train` keeps all its records. `Val` is split into `valid` and `test` with
`train_test_split`, stratified on the class folder so every class keeps the
same proportion in both sets.

**How to read the output:** the train/valid/test counts and the per-class
distribution. With `TEST_SIZE = 0.5`, the 510 validation records become about
255 validation and 255 test records.

In [ ]:
valid_records, test_records = train_test_split(
    val_source,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=[record["folder_class"] for record in val_source],
)

assignments = {
    "train": train_records,
    "valid": valid_records,
    "test": test_records,
}

for split, records in assignments.items():
    per_folder = collections.Counter(record["folder"] for record in records)
    print(f"{split:5s}: {len(records):4d} records {dict(per_folder)}")

print("Total assigned:", sum(len(records) for records in assignments.values()))

## 5. Copy every image with its annotation

Copies each pair from `data/archive/` to `data/yolo/images/{split}` and
`data/yolo/labels/{split}`, keeping the original file names so YOLO can match
them. Records whose image or label disappeared since notebook 05 are counted
as skipped instead of breaking the copy.

**How to read the output:** `train` + `valid` + `test` copied images must add
up to 5247 minus the skipped files, and the number of copied labels must be
identical to the number of copied images.

In [ ]:
copied = collections.Counter()
skipped = []
seen_stems = {split: set() for split in TARGET_SPLITS}

for split, records in assignments.items():
    for record in records:
        source_image = ROOT_DIR / record["image"]
        source_label = ROOT_DIR / record["label"]

        if not source_image.exists() or not source_label.exists():
            skipped.append({"split": split, "image": record["image"], "label": record["label"]})
            continue

        if record["stem"] in seen_stems[split]:
            raise FileExistsError(f"Duplicate file stem in {split}: {record['stem']}")
        seen_stems[split].add(record["stem"])

        shutil.copy2(source_image, OUTPUT_DIR / "images" / split / source_image.name)
        shutil.copy2(source_label, OUTPUT_DIR / "labels" / split / source_label.name)
        copied[split] += 1

print("Copied images per split:", dict(copied))
print("Copied total           :", sum(copied.values()))
print("Skipped files          :", len(skipped))
if skipped:
    display(pd.DataFrame(skipped))

## 6. Verify the copied dataset

Compares the number of images and labels per split and checks the pairing by
file stem again, this time on the output folder.

**How to read the output:** `unpaired` must be `0` for the three splits, and
the copied counts must match step 5. A non-zero value means an image was copied
without its label (or the opposite), which would make YOLO crash at training.

In [ ]:
verification = []
for split in TARGET_SPLITS:
    images = {
        path.stem
        for path in (OUTPUT_DIR / "images" / split).iterdir()
        if path.is_file()
    }
    labels = {
        path.stem
        for path in (OUTPUT_DIR / "labels" / split).iterdir()
        if path.is_file()
    }
    verification.append(
        {
            "split": split,
            "images": len(images),
            "labels": len(labels),
            "unpaired": len(images.symmetric_difference(labels)),
        }
    )

verification_df = pd.DataFrame(verification)
display(verification_df)

assert (verification_df["unpaired"] == 0).all(), "Some images/labels are not paired"
assert verification_df["images"].sum() == sum(copied.values())
print("Dataset copied and paired correctly.")

## 7. Write `data.yaml`

Creates the file Ultralytics uses to locate the dataset and to know the number
of classes and their names. `path` is written as an **absolute** path because
Ultralytics resolves relative roots against its own `datasets_dir` setting.

**How to read the output:** `nc` must be `4` and the three splits must point to
`images/train`, `images/valid` and `images/test`. The ids in `names` follow the
order used in the annotations (`0 = Glioma`, ..., `3 = Pituitary`).

In [ ]:
data_config = {
    "path": str(OUTPUT_DIR.resolve()),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": len(classes),
    "names": classes,
}

with open(DATA_YAML, "w") as handle:
    yaml.safe_dump(data_config, handle, sort_keys=False)

max_class_id = max(
    class_id
    for split in manifest["splits"]
    for record in manifest["splits"][split]
    for class_id in record["label_classes"]
)
assert max_class_id < data_config["nc"]

print(DATA_YAML.read_text())

## 8. Define the augmentation and optimization configuration

Stores the parameters used by `model.train(...)` in notebook 07 as
`data/yolo/train_config.json`. The values are the Ultralytics defaults adapted
to medical images:

- no vertical flip (`flipud = 0`) because it would reverse the anatomy;
- a small rotation (`degrees = 10`) and no perspective distortion;
- mosaic/mixup enabled to regularize a small dataset, with mosaic disabled
  for the last 10 epochs (`close_mosaic`);
- standard optimizer settings (AdamW/SGD chosen automatically by YOLO).

**How to read the output:** the table lists every parameter with its value and
its role. All of them are hyperparameters, not data: changing them only affects
the next training run.

In [ ]:
augmentation = {
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 10.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 2.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.1,
    "copy_paste": 0.0,
    "close_mosaic": 10,
}

optimization = {
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
}

explanations = {
    "hsv_h": "hue jitter — small, MRI contrast must not change",
    "hsv_s": "saturation jitter",
    "hsv_v": "brightness jitter",
    "degrees": "random rotation (deg), kept small for anatomy",
    "translate": "random translation (fraction of image)",
    "scale": "random zoom (fraction)",
    "shear": "random shear (deg)",
    "perspective": "random perspective, disabled for medical images",
    "flipud": "vertical flip, disabled (would reverse anatomy)",
    "fliplr": "horizontal flip probability",
    "mosaic": "mix 4 images into one, strong regularizer",
    "mixup": "blend two images and their boxes",
    "copy_paste": "segmentation-only, disabled",
    "close_mosaic": "disable mosaic for the last N epochs",
    "optimizer": "auto = AdamW or SGD chosen by Ultralytics",
    "lr0": "initial learning rate",
    "lrf": "final learning rate = lr0 * lrf",
    "momentum": "SGD momentum / Adam beta1",
    "weight_decay": "L2 regularization",
    "warmup_epochs": "epochs used to ramp up the learning rate",
}

rows = [
    {"section": "augmentation", "parameter": key, "value": value, "role": explanations[key]}
    for key, value in augmentation.items()
] + [
    {"section": "optimization", "parameter": key, "value": value, "role": explanations[key]}
    for key, value in optimization.items()
]
display(pd.DataFrame(rows))

train_config = {
    "created_by": "06_yolo_preprocessing.ipynb",
    "dataset": {
        "data_yaml": str(DATA_YAML),
        "classes": classes,
        "images_per_split": dict(copied),
        "test_size_from_val": TEST_SIZE,
        "seed": SEED,
    },
    "augmentation": augmentation,
    "optimization": optimization,
}

with open(TRAIN_CONFIG_PATH, "w") as handle:
    json.dump(train_config, handle, indent=2)

print("Training configuration saved:", TRAIN_CONFIG_PATH)

## 9. Visualize copied samples

Reads images and labels **from the new dataset** (not from the archive) and
draws the boxes for one image per class in each split.

**How to read the output:** the boxes must be at the same place as in notebook
05, and each split must contain the four classes. This proves that the copy
preserved both the images and their annotations.

In [ ]:
CLASS_COLORS = {0: "#e74c3c", 1: "#3498db", 2: "#2ecc71", 3: "#9b59b6"}


def read_boxes(label_path: Path):
    boxes = []
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        class_id, x, y, w, h = line.split()
        boxes.append((int(class_id), float(x), float(y), float(w), float(h)))
    return boxes


def draw_sample(axis, split: str, class_id: int):
    class_name = id_to_class[class_id]
    label_candidates = [
        label
        for label in sorted((OUTPUT_DIR / "labels" / split).glob("*.txt"))
        if any(
            int(line.split()[0]) == class_id
            for line in label.read_text().splitlines()
            if line.strip()
        )
    ]
    if not label_candidates:
        axis.axis("off")
        axis.set_title(f"{split} — {class_name} (none)", fontsize=9)
        return

    label_path = label_candidates[0]
    image_path = next((OUTPUT_DIR / "images" / split).glob(f"{label_path.stem}.*"))
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    axis.imshow(image)
    axis.axis("off")
    axis.set_title(f"{split} — {class_name}", fontsize=9)

    for cid, x, y, w, h in read_boxes(label_path):
        x1 = (x - w / 2) * width
        y1 = (y - h / 2) * height
        axis.add_patch(
            patches.Rectangle(
                (x1, y1),
                w * width,
                h * height,
                linewidth=2,
                edgecolor=CLASS_COLORS[cid],
                facecolor="none",
            )
        )
        axis.text(
            x1,
            max(y1 - 4, 8),
            id_to_class[cid],
            color="white",
            fontsize=7,
            bbox={"facecolor": CLASS_COLORS[cid], "pad": 1, "edgecolor": "none"},
        )


fig, axes = plt.subplots(len(TARGET_SPLITS), len(classes), figsize=(14, 10))
for row, split in enumerate(TARGET_SPLITS):
    for col, class_id in enumerate(range(len(classes))):
        draw_sample(axes[row, col], split, class_id)

plt.tight_layout()
plt.show()

## Next step

Open **`07_yolo_training.ipynb`**. It reads `data/yolo/data.yaml` and
`data/yolo/train_config.json`, chooses training hyperparameters adapted to the
available hardware (GPU or CPU), trains YOLOv8 and saves the best weights to
`models/yolo/`.